In [ ]:
import sys
import os
from jiwer import wer
import evaluate
import re
from models.whisper_transcriber import WhisperTranscriber, DanishTranscriber
from models.translator import ChunkTranslator
from utils.audio_preprocessing import preprocess_audio
from utils.audio_streaming import stream_audio
from utils.num_to_words import numbers_to_words

# Get the absolute path to the project root (parent directory of the current folder)
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))

# Add the project root to sys.path
sys.path.append(project_root)

### 1. Speech to text

In [ ]:
input_lang = "en"  
transcriber = WhisperTranscriber(min_chunk_duration_ms=4500) if input_lang=='en' else DanishTranscriber(min_chunk_duration_ms=4500)
translator = ChunkTranslator("Helsinki-NLP/opus-mt-en-da") if input_lang=='en' else ChunkTranslator("Helsinki-NLP/opus-mt-da-en")

speaker_num = 5
if input_lang == 'en':
    data_path = f"data/english/speaker_{speaker_num}_final.wav"
    data_path_txt = f"data/english/speaker_{speaker_num}_final.txt"
    data_path_translation = f"data/danish/dk_speaker_{speaker_num}_final.txt"
else:
    data_path = f"data/danish/dk_speaker_{speaker_num}.wav"
    data_path_txt = f"data/danish/dk_speaker_{speaker_num}.txt"
    data_path_translation = f"data/english/speaker_{speaker_num}_final.txt"


audio = preprocess_audio(data_path)
transcription_text = ""
translated_text = ""

# Get transcription and translation
for chunk in stream_audio(audio, frame_ms=200):
    transcription = transcriber.add_audio_chunk(chunk)
    if isinstance(transcription, str):
        transcription_text += " " + transcription
        translated = translator.translate_chunk(transcription)
        if isinstance(translated, str):
            translated_text += " " + translated


# At the end, flush any remaining audio
final_transcription = transcriber.flush()
transcription_text += " " + final_transcription
if isinstance(final_transcription, str):
    final_translation = translator.translate_chunk(final_transcription)
    if isinstance(final_translation, str):
        translated_text += " " + final_translation

translator.reset_context()

In [3]:
def clean_text(text):
    text = re.sub(r"-", " ", text)
    text = re.sub(r"[^\w\s]", "", text)
    text = text.lower()
    return text

transcription_text = clean_text(transcription_text)
translated_text = clean_text(translated_text)

In [4]:
transcription_text = numbers_to_words(transcription_text, lang='en', split_abbreviations=False)
translated_text = numbers_to_words(translated_text, lang='da', split_abbreviations=False)

In [ ]:
with open(data_path_txt, "r", encoding="utf-8") as f:
    ref_transcription = f.read()

wer_score = wer(reference=ref_transcription, hypothesis=transcription_text)
wer_score

### Text translation

In [6]:
with open(data_path_translation, "r", encoding="utf-8") as f:
    ref_translation = f.read()

In [ ]:
comet = evaluate.load("comet")
comet_score = comet.compute(
    predictions=[translated_text],
    references=[ref_translation],
    sources=[transcription_text],
)

comet_score

### Clean Danish text files (run 1 time)

In [9]:
# dk_txt_list = []
# for i in range(1, 11):
#     filename = f"data/danish/dk_speaker_{i}.txt"
#     with open(filename, "r", encoding="utf-8") as f:
#         dk_txt = f.read()
#         dk_txt_list.append(dk_txt)

In [10]:
# import os
# import re
# from utils.num_to_words import numbers_to_words

# os.makedirs("data/danish", exist_ok=True)

# for i, dk_txt in enumerate(dk_txt_list, start=1):
#     cleaned = clean_text(dk_txt)
#     converted = numbers_to_words(cleaned, lang='da', split_abbreviations=False)
#     out_filename = f"data/danish/dk_speaker_{i}_final.txt"
#     with open(out_filename, "w", encoding="utf-8") as f:
#         f.write(converted)